In [1]:
### Timing decorator
import time

def timing(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper


In [2]:
# Enable autoreload of imported modules

%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from pathlib import Path
import yaml
import spacy
@timing
def load_config(path):
    """
    Load event config file and data.

    Returns: Config `cfg` and DataFrame `df`.
    """
    config_file = path
    cfg = yaml.safe_load(Path(config_file).read_text(encoding='utf-8')).get('event', {})

    # Normalize keywords
    nlp = spacy.load("en_core_web_lg")
    doc = nlp(" ".join(cfg["keywords"]))
    cfg['keywords'] = [t.lemma_ for t in doc]

    df = pd.read_csv(cfg['input_file'])

    return cfg, df

/home/jupyter/miniconda3/envs/jupyter-lab/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
from preprocessing.normalize_text import normalize_text_series

@timing
def normalize_texts(df, batch_size=50, n_process=10):
    """
    Create clean text columns for 'title', 'text' and 'first_para'.
    """

    if 'title' in df:
        df['title_clean'] = normalize_text_series(df['title'], 
                                                  batch_size=batch_size, 
                                                  n_process=n_process)
    if 'text' in df:
        df['text_clean'] = normalize_text_series(df['text'], 
                                                  batch_size=batch_size, 
                                                  n_process=n_process)
    if 'first_para' in df:
        df['first_para_clean'] = normalize_text_series(df['first_para'], 
                                                  batch_size=batch_size, 
                                                  n_process=n_process)
    
    return df

In [5]:
from preprocessing.tfidf_dedupe import dedupe_tfidf_cosine

@timing
def dedupe_texts(df, text_col):
    """
    Dedupe entries in `df` with nearly identical values in `text_col`. Keeps the longest text by default.
    """
    threshold = 0.9
    min_df = 1  # min doc frequency
    ngram_min = 1
    ngram_max = 2
    max_features=None
    prefer_longer = True
    block_by_length = True  # compare only to similar length documents
    return_groups = True

    deduped_df, report = dedupe_tfidf_cosine(
        df,
        text_col=text_col,
        threshold=threshold,
        min_df=min_df,
        ngram_min=ngram_min,
        ngram_max=ngram_max,
        max_features=max_features,
        prefer_longer=prefer_longer,
        block_by_length=block_by_length,
        return_groups=return_groups,
    )

    return deduped_df, report

In [6]:
# Run text cleanup for all configs
from pathlib import Path
import pandas as pd
import yaml

config_path = Path("./config/")
config_files = [f for f in config_path.iterdir() if f.suffix=='.yaml']

for f in config_files:
    cfg = yaml.safe_load(Path(f).read_text(encoding='utf-8')).get('event', {})
    if not Path(cfg['input_file']).exists():
        print(f"{cfg['input_file']} not found")
        continue
    print(cfg)
    output_file = Path(str(cfg['input_file']).replace("/raw/", "/clean/"))
    if output_file.exists():
        print(f"Already exists: {output_file}")
        continue

    df = pd.read_csv(cfg['input_file'])
    df = normalize_texts(df)
    df, report_text = dedupe_texts(df, text_col='text_clean')
    df, report_title = dedupe_texts(df, text_col='title_clean')
    df, report_para = dedupe_texts(df, text_col='first_para')

    print(f"Documents after deduping: {len(df)}")
    print(f"Saving dataframe to: {output_file}")
    df.to_csv(output_file, index=None)

{'name': '2024_vehicleram_CA', 'state': 'CA', 'input_file': 'data/event_data/raw/2024_vehicleram_CA.csv', 'start_date': datetime.date(2024, 12, 24), 'end_date': datetime.date(2025, 2, 1), 'onset_date': datetime.date(2025, 1, 1), 'keywords': ['vehicle', 'ramming', 'attack', 'shooting', 'terrorism', 'crowd', 'injuries', 'suspect', 'kill', 'police']}
normalize_texts took 284.4435 seconds
dedupe_texts took 20.9121 seconds
dedupe_texts took 1.7297 seconds
dedupe_texts took 4.2341 seconds
Documents after deduping: 15941
Saving dataframe to: data/event_data/clean/2024_vehicleram_CA.csv
{'name': '2019_elpaso_tx', 'state': 'TX', 'input_file': 'data/event_data/raw/2019_elpaso_tx.csv', 'start_date': datetime.date(2019, 7, 26), 'end_date': datetime.date(2019, 9, 3), 'onset_date': datetime.date(2019, 8, 3), 'keywords': ['mass shooting', 'walmart', 'gunman', 'victims', 'terrorism', 'shooting', 'hate', 'racism', 'fatalities', 'gun violence']}
normalize_texts took 243.4000 seconds
dedupe_texts took 13

/tmp/ipykernel_965220/725730465.py:20: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cfg['input_file'])


normalize_texts took 83.0146 seconds
dedupe_texts took 4.0797 seconds
dedupe_texts took 0.3597 seconds
dedupe_texts took 0.6816 seconds
Documents after deduping: 4674
Saving dataframe to: data/event_data/clean/2019_vabeach_VA.csv
{'name': '2024_Helene_FL', 'input_file': 'data/event_data/raw/2024_Helene_FL.csv', 'start_date': datetime.date(2024, 9, 18), 'end_date': datetime.date(2024, 10, 29), 'onset_date': datetime.date(2024, 9, 25), 'keywords': ['hurricane', 'storm', 'flood', 'emergency', 'fatalities', 'helene', 'damage', 'response', 'landfall']}
normalize_texts took 237.5755 seconds
dedupe_texts took 11.7936 seconds
dedupe_texts took 1.1581 seconds
dedupe_texts took 1.8911 seconds
Documents after deduping: 10248
Saving dataframe to: data/event_data/clean/2024_Helene_FL.csv
{'name': '2022_uvalde_tx', 'state': 'TX', 'input_file': 'data/event_data/raw/2022_uvalde_tx.csv', 'start_date': datetime.date(2022, 5, 17), 'end_date': datetime.date(2022, 6, 24), 'onset_date': datetime.date(2022, 